# 500M-parameter / 20B-token code model — Colab pipeline

Replaces the 5B / 200M notebook. **Two parts, two different runtimes** (this saves compute units):

| Part | Runtime | What it does |
|---|---|---|
| **A. Build the corpus** | **CPU** | tokenizer + 8 sources → shards (synced to Drive) → merge → blended `corpus.bin` |
| **B. Train** | **H100 GPU** | restore corpus from Drive → train with automatic Drive backups → generate |

**Run the SETUP cell at the start of every session.** Colab wipes local disk when a session ends; everything that matters is copied to Drive
(`transformer_project_20b/`). Plan on roughly **100 GB of free Drive space** (shards ~40 GB + corpus ~40 GB + checkpoints).

**Before you start:** put `transformer-project-500m-20b.zip` in your Drive folder `MyDrive/transformer_project_20b/`, add `HF_TOKEN` to Colab *Secrets*,
and accept the terms of `bigcode/the-stack-dedup` on its Hugging Face page.

## SETUP — run at the start of EVERY session

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
import os, sys, json, shutil, threading, time

DRIVE   = "/content/drive/MyDrive/transformer_project_20b"
PROJECT = "/content/transformer-project"
os.makedirs(DRIVE, exist_ok=True)

ZIP = f"{DRIVE}/transformer-project-500m-20b.zip"
assert os.path.exists(ZIP), f"Upload transformer-project-500m-20b.zip to {DRIVE} first"
# Refresh the CODE from the zip on every run (so fixes you upload to Drive are picked up) but never touch artifacts/
_tmp = "/content/_unzip"; shutil.rmtree(_tmp, ignore_errors=True); shutil.unpack_archive(ZIP, _tmp)
_root = f"{_tmp}/transformer-project"; os.makedirs(PROJECT, exist_ok=True)
for item in os.listdir(_root):
    if item == "artifacts": continue
    dst = f"{PROJECT}/{item}"
    if os.path.isdir(f"{_root}/{item}"):
        shutil.rmtree(dst, ignore_errors=True); shutil.copytree(f"{_root}/{item}", dst)
    else:
        shutil.copy(f"{_root}/{item}", dst)
shutil.rmtree(_tmp)

!pip install -q -e {PROJECT} --no-deps
!pip install -q datasets tokenizers pyyaml tqdm rich

os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")
sys.path.insert(0, f"{PROJECT}/scripts")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])

def copy_verified(src, dst_dir):
    """Copy to a .part file, check the size, then rename — never leaves a half-written file under the real name."""
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, os.path.basename(src)); tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    assert os.path.getsize(tmp) == os.path.getsize(src), f"incomplete copy of {src}"
    os.replace(tmp, dst); return dst

def restore(src, dst):
    """Drive -> local disk, only if the local copy is missing."""
    if os.path.exists(dst): print("already local:", dst); return
    assert os.path.exists(src), f"not on Drive: {src}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    t = time.time(); shutil.copyfile(src, dst)
    print(f"restored {os.path.basename(dst)} ({os.path.getsize(dst)/1e9:.1f} GB, {time.time()-t:.0f}s)")

import transformer_project; print("project OK:", transformer_project.__file__)

---
# PART A — build the corpus (use a **CPU** runtime)

### A0. Stack Overflow Q&A (built directly from the posts data, not from a Hugging Face copy)

**Python plus language-neutral topics:** the build keeps questions tagged `python` (Python 2 ones dropped), and also questions about
SQL, regex, git, bash/linux, algorithms/data structures, docker, json and csv (full list: `EXTRA_TAGS` in the script) —
but only if they are *not* also tagged with another language (so "sql + java" is dropped, "python + sql" is kept).
Each row has a `bucket` field (`python` or `adjacent`) so you can see/reweight the split later.
`--extra-tags` with no values = Python only; `--tags --exclude-tags` = every language.

`scripts/fetch_stackoverflow.py` downloads the public per-year *posts* Parquet files (ClickHouse's conversion of the official Stack Exchange dump,
data up to **April 2024**, CC BY-SA 4.0), pairs each question with its accepted (or best-scoring) answer, converts HTML to text with fenced code blocks,
and writes `stackoverflow.jsonl`. It never touches the Stack Overflow website or API.

**Check the terms yourself before training on this.** Since July 2024 the *official* dumps sit behind a login with terms that exclude LLM training;
this older public copy is a different distribution, but whether you may use it for your purpose is your decision (and not legal advice from me).
The content is CC BY-SA: attribution and share-alike apply. Each row keeps the question id and year.

Run `--probe` first: it prints the real column names and one converted example. Then the full build (the download is large and takes a while;
each year's file is deleted after it is processed). Tune size with `--min-q-score`, `--min-a-score`, `--tags python` and `--sample-rate`.

In [ ]:
SO_LOCAL = "artifacts/stackoverflow.jsonl"
SO_DRIVE = f"{DRIVE}/stackoverflow.jsonl"

!python scripts/fetch_stackoverflow.py --years 2014 --probe --workdir /content/so_parquet --tags python

In [ ]:
REBUILD_SO = False     # set True ONCE after the tag filters change (a stackoverflow.jsonl built with the old, leakier
                       # filter may still be on Drive); it deletes the old file so it is rebuilt - then set it back to False
if REBUILD_SO:
    for f in (SO_LOCAL, SO_DRIVE):
        if os.path.exists(f): os.remove(f); print("deleted", f)

if os.path.exists(SO_DRIVE):
    restore(SO_DRIVE, SO_LOCAL)
else:
    !python scripts/fetch_stackoverflow.py --years 2008-2024 --out {SO_LOCAL} --workdir /content/so_parquet \
        --min-q-score 1 --min-a-score 1 --tags python --exclude-tags python-2.7 python-2.x python-2
    copy_verified(SO_LOCAL, DRIVE)
print(f"{os.path.getsize(SO_LOCAL)/1e9:.1f} GB of Stack Overflow Q&A text")

### A1. Choose sources, then check them

`python_edu` is skipped by default: the Hub copy only lists file ids, the code itself has to be downloaded from the Software Heritage bucket.
If you do that, save it as jsonl (`{"content": "..."}` per line), put it in Drive, remove it from `SKIP` and add it to `LOCAL`.
The plan prints the resulting mix (ratios are re-normalised over what you keep).

In [ ]:
SKIP  = ["python_edu", "extras"]  # sources to leave out. extras = GitHub issues from ALL languages (no Python filter)
LOCAL = {"stackoverflow": SO_LOCAL}   # sources supplied as local jsonl. python_edu would go here too, e.g.
                                       # {"python_edu": "/content/drive/MyDrive/python_edu.jsonl"}

skip_arg  = f"--skip {' '.join(SKIP)}" if SKIP else ""
local_arg = " ".join(f"--local {k}={v}" for k, v in LOCAL.items())
!python scripts/build_corpus_20b.py plan {skip_arg}

**`probe` fetches one row from every source and prints its fields** (one row, no shuffle buffer, so it is cheap on RAM). Run it and read the output — dataset names and schemas change, and a wrong guess found here costs seconds instead of a day.

In [ ]:
!python scripts/build_corpus_20b.py probe {skip_arg} {local_arg}

### A2. Tokenizer for the new mix (32k vocab, with the new structural tokens)

**ALREADY DONE - do not re-run this step.** The tokenizer your model was trained with is on Drive; this cell now refuses to retrain it (and only prints a check that the file is fine).

In [ ]:
# ============================================================================
# A2 is ONLY for the very first build. Your model was trained with the tokenizer that already
# exists on Drive; a re-trained tokenizer would NOT match the model's learned token numbers.
# So this cell refuses to retrain while the Drive copy exists.
# ============================================================================
RETRAIN_TOKENIZER = False   # leave False. (Even if True, the result is saved under a NEW name and never replaces the original.)
TOK_DRIVE = f"{DRIVE}/tokenizer_20b_32k.json"

if os.path.exists(TOK_DRIVE) and not RETRAIN_TOKENIZER:
    restore(TOK_DRIVE, "artifacts/tokenizer_20b_32k.json")
    from tokenizers import Tokenizer
    _t = Tokenizer.from_file("artifacts/tokenizer_20b_32k.json")
    print(f"Tokenizer already exists on Drive - NOT retraining. vocab={_t.get_vocab_size()} | "
          f"<file_sep>={_t.token_to_id('<file_sep>')} <question>={_t.token_to_id('<question>')} (a number = the special tokens are there)")
else:
    # "&&" = the tokenizer is only trained if sampling succeeded (separate "!" lines would carry on after an error)
    !rm -rf artifacts/tok_sample && python scripts/build_corpus_20b.py sample --mb 800 --out artifacts/tok_sample {skip_arg} {local_arg} && tp-train-tokenizer --files artifacts/tok_sample/*.txt --vocab-size 32000 --out artifacts/tokenizer_NEW.json
    assert os.path.exists("artifacts/tokenizer_NEW.json"), "tokenizer training did not finish - see the error above"
    copy_verified("artifacts/tokenizer_NEW.json", DRIVE)
    print("saved as tokenizer_NEW.json - the original tokenizer_20b_32k.json was NOT touched")

### Recovery — only if your runtime was reset or logged out mid-build

Run **SETUP** first, then this cell. It restores the tokenizer from Drive, redefines the variables the later cells need, and **checks what is safely on Drive** for each source
(finished shards vs. what the progress file claims). Then simply re-run the A3 loop below: finished sources are skipped in seconds and the unfinished one resumes after its last finished shard.
A resumed source first has to stream past the rows it already did (it prints its progress while doing so), so allow some extra minutes before new tokens appear.

In [ ]:
import glob
restore(f"{DRIVE}/tokenizer_20b_32k.json", "artifacts/tokenizer_20b_32k.json")

SO_LOCAL = "artifacts/stackoverflow.jsonl"
SKIP  = ["python_edu", "extras"]
LOCAL = {"stackoverflow": SO_LOCAL}
skip_arg  = f"--skip {' '.join(SKIP)}" if SKIP else ""
local_arg = " ".join(f"--local {k}={v}" for k, v in LOCAL.items())
ORDER = [n for n in ["code_contests", "apps", "wikipedia", "math", "fineweb_edu", "stackoverflow", "commits",
                     "extras", "python_edu", "raw_python"] if n not in SKIP]

print(f"{'source':<15}{'state':<10}{'tokens (progress)':>20}{'tokens on Drive':>20}  check")
for name in ORDER:
    d = f"{DRIVE}/src_20b/{name}"
    if not os.path.exists(f"{d}/progress.json"):
        print(f"{name:<15}{'not started':<10}"); continue
    prog = json.load(open(f"{d}/progress.json"))
    on_drive = sum(os.path.getsize(f) for f in glob.glob(f"{d}/shard_*.bin")) // 2
    print(f"{name:<15}{'DONE' if prog['done'] else 'partial':<10}{prog['tokens']:>20,}{on_drive:>20,}  "
          f"{'OK' if on_drive == prog['tokens'] else 'MISMATCH - send this line to Claude'}")

### A3. Tokenize every source (resumable)

Each source is written as ~1B-token shards; every finished shard is copied to Drive together with a progress file.
If the runtime dies, **just run this cell again** — it continues after the last finished shard (at most ~1B tokens are redone).
Smallest sources first so you see results early; `raw_python` (10B) is last and slowest.
**Short on time?** `raw_python` can be interrupted at any point (the stop button): every finished ~1B-token shard is already on Drive.
Then set `ALLOW_PARTIAL = True` in A4 and carry on — the corpus is simply smaller (fewer raw-Python tokens). Re-running this cell later continues where it stopped.

In [ ]:
ORDER = [n for n in ["code_contests", "apps", "wikipedia", "math", "fineweb_edu", "stackoverflow", "commits",
                     "extras", "python_edu", "raw_python"] if n not in SKIP]
for name in ORDER:
    print("=" * 70, "\n", name)
    !python scripts/build_corpus_20b.py tokenize --tokenizer artifacts/tokenizer_20b_32k.json \
        --out artifacts/src_20b --sync-dir {DRIVE}/src_20b --only {name} {local_arg}

### A4. Merge shards into one `.bin` per source, and check how many tokens each really has

In [ ]:
import build_corpus_20b as bc
ALLOW_PARTIAL = False      # True if you STOPPED a source early (e.g. interrupted raw_python): uses only its finished ~1B-token shards
partial_arg = "--allow-partial" if ALLOW_PARTIAL else ""
for name in ORDER:
    !python scripts/build_corpus_20b.py merge --out artifacts/src_20b --sync-dir {DRIVE}/src_20b --only {name} {partial_arg}
    shutil.rmtree(f"artifacts/src_20b/{name}", ignore_errors=True)     # shards stay safe on Drive; frees local disk

print(f"\n{'source':<15}{'available':>16}{'budget':>16}{'fill':>7}")
for s in bc.SOURCES:
    if s.name in SKIP: continue
    n = os.path.getsize(f"artifacts/src_20b/{s.name}.bin") // 2
    print(f"{s.name:<15}{n:>16,}{s.tokens:>16,}{n/s.tokens:>7.0%}")
print("\nSources well below 100% cannot supply their budget. The next cell caps each one to what it really has (plus a limited number of repeats).")

### A5. Blend into the final corpus and back it up

**How the sizes are decided:** each source contributes `min(budget, available x allowed epochs)` tokens. Big sources (raw Python, fineweb, wikipedia, math) get **1 epoch = no repeats**.
Small ones may be repeated a few times (`EPOCH_CAP`). The corpus is exactly the sum of what the sources contribute, and the table shows the repeat count for each.
Repeating a tiny source many times just teaches the model to memorise it, so keep the caps small.

In [ ]:
EPOCH_CAP = {"code_contests": 3, "apps": 3, "commits": 3, "stackoverflow": 2}     # any source not listed: 1 (no repeats)

srcs  = [s for s in bc.SOURCES if s.name not in SKIP]
avail = {s.name: os.path.getsize(f"artifacts/src_20b/{s.name}.bin") // 2 for s in srcs}
use   = {s.name: int(min(s.tokens, avail[s.name] * EPOCH_CAP.get(s.name, 1))) for s in srcs}
total = sum(use.values())

print(f"{'source':<15}{'available':>15}{'used':>15}{'epochs':>8}{'share':>8}")
for s in srcs:
    print(f"{s.name:<15}{avail[s.name]:>15,}{use[s.name]:>15,}{use[s.name]/avail[s.name]:>8.2f}{use[s.name]/total:>8.1%}")
print(f"{'TOTAL':<15}{'':>15}{total:>15,}")

source_args = " ".join(f"--source artifacts/src_20b/{n}.bin:{use[n]/total:.6f}" for n in use)
!rm -f artifacts/mix_20b/corpus.bin artifacts/mix_20b/corpus.manifest.json
!python scripts/build_data_mix.py --dtype uint16 --context-len 2048 --seed 42 \
    --total-tokens {total} --max-repeat 3.01 --out artifacts/mix_20b/corpus.bin {source_args}

for f in ["corpus.bin", "corpus.manifest.json"]:
    copy_verified(f"artifacts/mix_20b/{f}", f"{DRIVE}/mix_20b")
print("corpus backed up to Drive")

Part A is done. When the corpus is on Drive you can **switch the runtime to H100** (Runtime → Change runtime type) and continue with Part B. Run SETUP again first.

---
# PART B — train (use the **H100** runtime)

Run **SETUP** first (it is a fresh machine). Then restore the corpus, create the run config, train.

### B0. Smoke test (a minute; run once per fresh H100 session, BEFORE the long run)

The attention layer now uses `is_causal=True` (fused FlashAttention path, no per-layer mask buffer) instead of an explicit mask.
This cell proves on the real GPU that the two are numerically identical and runs the project's test-suite. **If anything fails here, stop and tell me.**

In [ ]:
!pip install -q pytest
import torch, torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"
q = torch.randn(2, 4, 256, 64, device=dev); k = torch.randn_like(q); v = torch.randn_like(q)
mask = torch.tril(torch.ones(256, 256, dtype=torch.bool, device=dev))
a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
b = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("is_causal == explicit mask:", torch.allclose(a, b, atol=1e-5), "| max diff", (a - b).abs().max().item())
!python -m pytest -q -x tests

### B1. Restore corpus + tokenizer from Drive to local disk (~40 GB, several minutes)

In [ ]:
FORCE_RESTORE = False   # set True ONCE if you REBUILT the corpus on Drive: otherwise an old local copy in this session would be reused
if FORCE_RESTORE:
    for f in ("corpus.bin", "corpus.manifest.json"):
        if os.path.exists(f"artifacts/mix_20b/{f}"): os.remove(f"artifacts/mix_20b/{f}"); print("deleted old local", f)

restore(f"{DRIVE}/mix_20b/corpus.bin",           "artifacts/mix_20b/corpus.bin")
restore(f"{DRIVE}/mix_20b/corpus.manifest.json", "artifacts/mix_20b/corpus.manifest.json")
restore(f"{DRIVE}/tokenizer_20b_32k.json",       "artifacts/tokenizer_20b_32k.json")

manifest = json.load(open("artifacts/mix_20b/corpus.manifest.json"))
n = os.path.getsize("artifacts/mix_20b/corpus.bin") // 2
assert n == manifest["total_tokens"], "corpus.bin is incomplete or corrupt — re-copy it"
print(f"corpus OK: {n:,} tokens")
!nvidia-smi --query-gpu=name,memory.total --format=csv

### B2. Run config

* `MICRO_BATCH = 8` is the safe default for an 80 GB card with a 2048 context (the optimizer step always stays at 524,288 tokens;
  `grad_accum` adjusts). If you want to try 16 and it fits, watch the first steps for an out-of-memory error.
* `TOKENS_TO_TRAIN = None` trains on the whole corpus. Set e.g. `4_000_000_000` for a first, shorter run.
  The learning-rate schedule is tied to the total step count, so a short run ends fully decayed — it is a complete small experiment,
  not a pause you can extend later without restarting the schedule.
* The finished config is **saved to Drive and reused on resume** — changing batch size or step count between sessions would break the resume.

In [ ]:
MICRO_BATCH     = 8
TOKENS_TO_TRAIN = None        # None = whole corpus; or e.g. 4_000_000_000

from transformer_project.config import RunConfig
RUN_YAML = "configs/run_stage.yaml"
RUN_YAML_DRIVE = f"{DRIVE}/run_stage.yaml"

RESET_RUN = False   # set True ONCE if the corpus or settings changed (e.g. you rebuilt the corpus): deletes the saved run config
                    # (it has the OLD max_steps in it) and the old checkpoints, so training restarts cleanly. Then set it back to False.
if RESET_RUN:
    for f in (RUN_YAML_DRIVE, RUN_YAML, f"{DRIVE}/run/ckpt_latest.pt", f"{DRIVE}/run/ckpt_best.pt"):
        if os.path.exists(f): os.remove(f); print("deleted", f)
    shutil.rmtree("artifacts/run_500m_20b", ignore_errors=True)

if os.path.exists(RUN_YAML_DRIVE):
    shutil.copy(RUN_YAML_DRIVE, RUN_YAML)
    print("Using the run config saved on Drive (keeps a resumed run consistent).")
else:
    cfg = RunConfig.from_yaml("configs/code_500m_20b.yaml")
    ctx = cfg.model.context_len
    cfg.train.batch_size = MICRO_BATCH
    cfg.train.grad_accum = 524_288 // (MICRO_BATCH * ctx)
    tokens_per_step = cfg.train.batch_size * cfg.train.grad_accum * ctx
    budget = manifest["total_tokens"] if TOKENS_TO_TRAIN is None else min(TOKENS_TO_TRAIN, manifest["total_tokens"])
    cfg.train.max_steps  = budget // tokens_per_step
    cfg.train.save_every = 250
    cfg.train.log_every  = 10        # one line every ~10 steps (at ~16 s/step: every ~3 min) so it never looks stuck
    cfg.train.eval_every = 500
    cfg.to_yaml(RUN_YAML)
    shutil.copy(RUN_YAML, RUN_YAML_DRIVE)
    print(f"{cfg.train.max_steps:,} steps x {tokens_per_step:,} tokens = {cfg.train.max_steps*tokens_per_step/1e9:.2f}B tokens")

cfg = RunConfig.from_yaml(RUN_YAML)
LOCAL_RUN = cfg.train.output_dir
print("micro-batch", cfg.train.batch_size, "| grad_accum", cfg.train.grad_accum, "| max_steps", cfg.train.max_steps, "| output:", LOCAL_RUN)

### B3. Train

A background thread copies `ckpt_latest.pt` / `ckpt_best.pt` to Drive every 10 minutes (only when they changed).
**If the session dies:** open a new H100 session, run SETUP → B1 → B2 (with `RESET_RUN = False`!) → this cell. It restores the latest checkpoint from Drive and resumes; if Drive has checkpoints but nothing can be restored it stops instead of starting from scratch. Drive keeps the newest checkpoint plus one previous copy (~6 GB each).
On resume the trainer replays the already-used batches to continue the data order — that can take several minutes at a late step.

In [ ]:
DRIVE_RUN = f"{DRIVE}/run"             # just a path; folders are only created when a file is copied into them
CK = f"{LOCAL_RUN}/checkpoints"

# 1) fresh machine: bring the newest checkpoint back from Drive (falls back to the previous copy if the newest is missing)
if not os.path.exists(f"{CK}/ckpt_latest.pt"):
    src = next((f"{DRIVE_RUN}/{n}" for n in ("ckpt_latest.pt", "ckpt_latest.prev.pt") if os.path.exists(f"{DRIVE_RUN}/{n}")), None)
    if src: restore(src, f"{CK}/ckpt_latest.pt")

# 2) SAFETY: never start from scratch when Drive already holds checkpoints. A from-scratch run would overwrite the good
#    ckpt_latest.pt on Drive with a step-250 one at its first save.
if not os.path.exists(f"{CK}/ckpt_latest.pt"):
    existing = [f for f in ("ckpt_latest.pt", "ckpt_latest.prev.pt", "ckpt_best.pt") if os.path.exists(f"{DRIVE_RUN}/{f}")]
    assert not existing, f"Drive has {existing} but nothing could be restored - NOT starting from scratch. Look into this first."
resume_arg = f"--resume {CK}/ckpt_latest.pt" if os.path.exists(f"{CK}/ckpt_latest.pt") else ""
print("RESUMING from checkpoint" if resume_arg else "no checkpoint anywhere: STARTING FROM SCRATCH (correct only for the very first run)")

# 3) background Drive backup, keeping ONE previous version of ckpt_latest.pt as a safety net
SYNC_STOP, _last = threading.Event(), {}
def backup_checkpoints():
    for name in ["ckpt_latest.pt", "ckpt_best.pt", "ckpt_final.pt"]:
        src = f"{CK}/{name}"
        if os.path.exists(src):
            key = (os.stat(src).st_mtime, os.stat(src).st_size)
            if _last.get(name) != key:
                try:
                    dst = f"{DRIVE_RUN}/{name}"
                    if name == "ckpt_latest.pt" and os.path.exists(dst):
                        os.replace(dst, f"{DRIVE_RUN}/ckpt_latest.prev.pt")
                    copy_verified(src, DRIVE_RUN); _last[name] = key
                except Exception as e: print("checkpoint backup failed (will retry):", e)
def sync_loop():
    while not SYNC_STOP.wait(600): backup_checkpoints()
threading.Thread(target=sync_loop, daemon=True).start()

try:
    !tp-train --config {RUN_YAML} {resume_arg}
finally:
    SYNC_STOP.set()
    backup_checkpoints()                      # final copy even if training stopped early
    print("checkpoints backed up to", DRIVE_RUN)

### B4. Try the model

In [ ]:
!tp-generate --config {RUN_YAML} --checkpoint {CK}/ckpt_best.pt \
    --prompt 'def fibonacci(n):' --strategy sample --max-new-tokens 200 --temperature 0.8 --top-k 50 --top-p 0.95

In [ ]:
# Prompt built in Python so the newlines are real newlines (shell quoting would send a literal backslash-n)
import subprocess
prompt = "<commit_before>def add(a, b):\n    return a - b\n<commit_message>"
r = subprocess.run(["tp-generate", "--config", RUN_YAML, "--checkpoint", f"{CK}/ckpt_best.pt",
                    "--prompt", prompt, "--strategy", "greedy", "--max-new-tokens", "80"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])

---
# PART D — fine-tune: follow the prompt, stop when done, edit code (with pretraining data mixed back in)

**What this does.** Continues from your finished model with a *short, low-learning-rate* run on ~55k examples built from data you already have:
function bodies from real code, Stack Overflow answers, small commit edits and contest solutions. The formats are exactly the ones used in pretraining;
what changes is that **the loss is only on the response** and every response ends with an end-of-text token, so the model learns **where to stop**.
**25% of every batch is ordinary pretraining data** (all tokens, as before), so it keeps what it learned.

| Step | Runtime | Time (my estimate) |
|---|---|---|
| D1 build the pairs | CPU | ~10-40 min (streams from Hugging Face) |
| D2 restore + settings | H100 | a few min (copies the ~32 GB corpus for the replay share) |
| D3 train | H100 | ~1.3 h per epoch at the speed you measured; 2 epochs ~2.6 h |
| D4 compare base vs fine-tuned | H100 | minutes |

Your base checkpoint is **never modified**: the new run writes to its own folder.

### D0. Before anything else: make sure the FINAL base checkpoint is on Drive (run in the session that finished training)

In [ ]:
# Only ckpt_latest/ckpt_best were being backed up. Copy the final one now, while the local file still exists:
_f = f"{LOCAL_RUN}/checkpoints/ckpt_final.pt"
if os.path.exists(_f): copy_verified(_f, f"{DRIVE}/run"); print("backed up", _f)
else: print("no local ckpt_final.pt in this session (fine if it is already on Drive):", os.path.exists(f"{DRIVE}/run/ckpt_final.pt"))

### D1. Build the fine-tuning pairs (CPU runtime; run SETUP first)

**Data quality rules (new):** function examples are kept only if they have a real docstring and their body uses nothing defined elsewhere in the original file (builtins, parameters, and the imports shown in the prompt only). Without this rule the model learned to invent undefined names such as `EMAIL_REGEX`.

In [ ]:
SO_LOCAL = "artifacts/stackoverflow.jsonl"
restore(f"{DRIVE}/stackoverflow.jsonl", SO_LOCAL)
!python scripts/build_sft_data.py --so-jsonl {SO_LOCAL} --out artifacts/sft/sft_pairs.jsonl \
    --n-functions 25000 --n-so 15000 --n-commits 10000 --n-contests 5000
copy_verified("artifacts/sft/sft_pairs.jsonl", f"{DRIVE}/sft")

Read the printed examples: each kind shows the **tail of a prompt** and the **head of a response**. Function prompts end at the signature/docstring (or a newline), Q&A prompts end with `<answer>`, commit prompts end with `<commit_after>`.

### D2. Restore everything and set the number of steps (H100 runtime; run SETUP first)

`EPOCHS = 2` means every fine-tuning example is seen twice. Fewer passes = gentler. More than ~3 tends to make a model memorise the examples.

In [ ]:
EPOCHS = 2

restore(f"{DRIVE}/sft/sft_pairs.jsonl",     "artifacts/sft/sft_pairs.jsonl")
restore(f"{DRIVE}/mix_20b/corpus.bin",      "artifacts/mix_20b/corpus.bin")          # the replay share
restore(f"{DRIVE}/tokenizer_20b_32k.json",  "artifacts/tokenizer_20b_32k.json")
_base = next((f"{DRIVE}/run/{n}" for n in ("ckpt_final.pt", "ckpt_latest.pt") if os.path.exists(f"{DRIVE}/run/{n}")), None)
assert _base, "no base checkpoint on Drive (run D0 in the session that trained the model)"
restore(_base, "artifacts/base/base.pt")
print("base checkpoint:", _base)

from transformer_project.config import RunConfig
cfg = RunConfig.from_yaml("configs/sft_500m.yaml")
n_pairs = sum(1 for _ in open(cfg.data.train_files[0]))
train_pairs = int(n_pairs * (1 - cfg.data.val_fraction))
items = train_pairs / (1 - cfg.data.replay_fraction)                     # fine-tuning pairs + replay chunks per epoch
seqs_per_step = cfg.train.batch_size * cfg.train.grad_accum
cfg.train.max_steps = EPOCHS * int(items // seqs_per_step)
cfg.to_yaml("configs/run_sft.yaml")
hours = cfg.train.max_steps * seqs_per_step * cfg.model.context_len / 32_000 / 3600
print(f"{n_pairs:,} pairs -> {train_pairs:,} train + {items - train_pairs:,.0f} replay chunks per epoch")
print(f"{cfg.train.max_steps:,} steps x {seqs_per_step} sequences | about {hours:.1f} h at ~32k tokens/s (padded to {cfg.model.context_len})")

### D3. Train

Checkpoints go to their own folder and are copied to Drive every 10 minutes. After a disconnect: SETUP → D2 → this cell; it resumes from the last checkpoint.
The first lines show **loss on the validation pairs** (answer tokens only): it should fall quickly from where the base model starts, then flatten.

In [ ]:
SFT_CK, SFT_DRIVE = f"{cfg.train.output_dir}/checkpoints", f"{DRIVE}/sft_run"
RESTART_FINETUNE = False   # set True ONCE after you rebuilt the data (D1): throws away the earlier fine-tuning run so this one starts
                           # again from the base model. Otherwise an old finished run on Drive would be RESUMED instead. Then set it back to False.
if RESTART_FINETUNE:
    shutil.rmtree(cfg.train.output_dir, ignore_errors=True)
    for n in ("ckpt_latest.pt", "ckpt_best.pt", "ckpt_final.pt"):
        f = f"{SFT_DRIVE}/{n}"
        if os.path.exists(f): os.remove(f); print("deleted", f)
if not os.path.exists(f"{SFT_CK}/ckpt_latest.pt") and os.path.exists(f"{SFT_DRIVE}/ckpt_latest.pt"):
    restore(f"{SFT_DRIVE}/ckpt_latest.pt", f"{SFT_CK}/ckpt_latest.pt")
if os.path.exists(f"{SFT_CK}/ckpt_latest.pt"):
    start_arg = f"--resume {SFT_CK}/ckpt_latest.pt"; print("RESUMING the fine-tuning run")
else:
    start_arg = "--init-from artifacts/base/base.pt"; print("fresh fine-tuning run from the base model")

_stop, _seen = threading.Event(), {}
def _backup():
    for n in ("ckpt_latest.pt", "ckpt_best.pt", "ckpt_final.pt"):
        f = f"{SFT_CK}/{n}"
        if os.path.exists(f):
            key = (os.stat(f).st_mtime, os.stat(f).st_size)
            if _seen.get(n) != key:
                try: copy_verified(f, SFT_DRIVE); _seen[n] = key
                except Exception as e: print("backup failed (will retry):", e)
threading.Thread(target=lambda: [_backup() for _ in iter(lambda: _stop.wait(600), True)], daemon=True).start()
try:
    !tp-train --config configs/run_sft.yaml {start_arg}
finally:
    _stop.set(); _backup(); print("fine-tuned checkpoints backed up to", SFT_DRIVE)

### D4. Compare the base model and the fine-tuned model on the same prompts

Prompts end where a *response* should begin (the closing quotes of a docstring, `<answer>`, `<commit_after>`), matching how the model was fine-tuned.
The fine-tuned model should stop on its own; the base model needs the output trimmed after the first function.

In [ ]:
import subprocess, re
LOG = re.compile(r"^\d\d:\d\d:\d\d \| (INFO|WARNING|ERROR|DEBUG)")

def gen(prompt, ckpt, cfg_path="configs/run_sft.yaml", max_new_tokens=200, temperature=0.2):
    r = subprocess.run(["tp-generate", "--config", cfg_path, "--checkpoint", ckpt, "--prompt", prompt,
                        "--strategy", "sample", "--temperature", str(temperature), "--top-k", "50",
                        "--max-new-tokens", str(max_new_tokens)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout or r.stderr).splitlines() if not LOG.match(l))

tests = {
 "function":    '<file_sep>import re\n\n\ndef extract_emails(text: str) -> list:\n    """Return all email addresses found in text."""',
 "q&a":         '<question>How do I merge two dictionaries in Python?<answer>',
 "edit":        '<commit_before>def mean(xs):\n    return sum(xs) / (len(xs) - 1)\n<commit_message>Fix off-by-one in mean<commit_after>',
 "contest":     '<question>Read two integers a and b from one line and print their sum.<answer>',
}
SFT_BEST = f"{cfg.train.output_dir}/checkpoints/ckpt_final.pt"
if not os.path.exists(SFT_BEST): SFT_BEST = f"{cfg.train.output_dir}/checkpoints/ckpt_best.pt"
for name, p in tests.items():
    print(f"################ {name} ################")
    print("--- BASE ---\n" + gen(p, "artifacts/base/base.pt")[:900])
    print("--- FINE-TUNED ---\n" + gen(p, SFT_BEST)[:900] + "\n")

---
# PART E — fine-tune on ~2 billion tokens (packed, loss on responses only, with pretraining data mixed back in)

Part D uses ~55k examples, each padded to a full 2048-token window (about 27% of the compute does real work). At 2B tokens that waste would be unaffordable, so this part **packs several short examples into each row** (about 99% of the row is real data), and loss is still **only on the response** plus the end-of-text token.

| Kind | Real tokens (default) | What it teaches |
|---|---|---|
| function bodies | 0.30B | signature + docstring → body, and **where a function ends** |
| Stack Overflow | 0.45B | question → answer (Python and adjacent topics) |
| small commit edits | 0.15B | old file + message → new file (actually changed) |
| contest solutions | 0.06B | problem → correct Python solution |
| **pretraining replay** | ~20% of every batch | keeps what the base model already knows |

The small sources (commits, contests, Stack Overflow) are repeated up to 3 passes if they run short; the builder reports exactly what it got. **The result is ~1.2B tokens per pass, about 10-11 GPU hours, ~2,300 steps** (my estimate at the speed you measured: ~95-115 compute units). Function bodies are only kept when they are **self-contained** (they use nothing that is defined elsewhere in their original file) and have a docstring; that removes most functions, which is why this part is smaller than before and why quality matters more than size here.

Your base checkpoint is never modified. Part D remains the quick version; use E if you want the larger run.

### E1. Build the packed data (CPU runtime; run SETUP first)

Streams about 1-2M Python files from Hugging Face, tokenizes and packs. **Estimate: 1-4 hours** (the slow part is parsing files, not tokenizing). Progress lines show tokens packed per kind. Everything is copied to Drive at the end.

In [ ]:
SO_LOCAL = "artifacts/stackoverflow.jsonl"
restore(f"{DRIVE}/stackoverflow.jsonl", SO_LOCAL)
restore(f"{DRIVE}/tokenizer_20b_32k.json", "artifacts/tokenizer_20b_32k.json")

!python scripts/build_sft_packed.py --tokenizer artifacts/tokenizer_20b_32k.json --so-jsonl {SO_LOCAL} \
    --out artifacts/sft_packed/sft2b --tokens-functions 0.3e9 --tokens-so 0.45e9 --tokens-commits 0.15e9 --tokens-contests 0.06e9

import glob
for f in glob.glob("artifacts/sft_packed/sft2b.*"):
    copy_verified(f, f"{DRIVE}/sft_packed")
print("copied to Drive:", sorted(os.listdir(f"{DRIVE}/sft_packed")))

**Read the summary the builder prints.** For each kind it shows the tokens it actually produced; a line ending in `<-- short of its ... budget` means that source ran out (small sources) or the stream ended. That is fine, the total is just smaller. The line `NN% of tokens carry loss` is the share of response tokens (typically 50-70%).
If `fill` is below ~95% something is off; send me the summary.

### E2. Restore everything and set the number of steps (H100 runtime; run SETUP first)

In [ ]:
EPOCHS = 1      # passes over the packed data; 1 is plenty at this size

for f in ("train.tokens.bin", "train.mask.bin", "val.tokens.bin", "val.mask.bin", "meta.json"):
    restore(f"{DRIVE}/sft_packed/sft2b.{f}", f"artifacts/sft_packed/sft2b.{f}")
restore(f"{DRIVE}/mix_20b/corpus.bin",     "artifacts/mix_20b/corpus.bin")          # the replay share
restore(f"{DRIVE}/tokenizer_20b_32k.json", "artifacts/tokenizer_20b_32k.json")
_base = next((f"{DRIVE}/run/{n}" for n in ("ckpt_final.pt", "ckpt_latest.pt") if os.path.exists(f"{DRIVE}/run/{n}")), None)
assert _base, "no base checkpoint on Drive (run D0 in the session that trained the model)"
restore(_base, "artifacts/base/base.pt")
print("base checkpoint:", _base)

meta = json.load(open("artifacts/sft_packed/sft2b.meta.json"))
print("packed data:", {k: (f"{v:,}" if isinstance(v, int) else round(v, 3)) for k, v in meta.items() if k in ("rows_train", "rows_val", "real_tokens", "fill_rate")})
print("tokens per kind:", {k: f"{v/1e6:,.0f}M" for k, v in meta["tokens_per_kind"].items()})

from transformer_project.config import RunConfig
cfg = RunConfig.from_yaml("configs/sft2b_500m.yaml")
items = meta["rows_train"] / (1 - cfg.data.replay_fraction)                 # packed rows + replay rows per epoch
seqs_per_step = cfg.train.batch_size * cfg.train.grad_accum
cfg.train.max_steps = EPOCHS * int(items // seqs_per_step)
cfg.to_yaml("configs/run_sft2b.yaml")
tokens = cfg.train.max_steps * seqs_per_step * cfg.model.context_len
hours = tokens / 32_000 / 3600
print(f"{cfg.train.max_steps:,} steps x {seqs_per_step} rows | {tokens/1e9:.2f}B tokens | about {hours:.0f} h at ~32k tokens/s "
      f"(~{hours*8.9:,.0f} units at 8.9/h, ~{hours*10.8:,.0f} at 10.8/h)")

### E3. Train

Same safety net as before: checkpoints are copied to Drive every 10 minutes; after a disconnect run SETUP → E2 → this cell and it **resumes**. The validation loss is on held-out packed rows (answer tokens only): it should fall quickly in the first few hundred steps and then flatten.

In [ ]:
S2_CK, S2_DRIVE = f"{cfg.train.output_dir}/checkpoints", f"{DRIVE}/sft2b_run"
RESTART_FINETUNE = False   # set True ONCE after you rebuilt the data (E1): discards an earlier run so this starts from the base model
if RESTART_FINETUNE:
    shutil.rmtree(cfg.train.output_dir, ignore_errors=True)
    for n in ("ckpt_latest.pt", "ckpt_best.pt", "ckpt_final.pt"):
        f = f"{S2_DRIVE}/{n}"
        if os.path.exists(f): os.remove(f); print("deleted", f)
if not os.path.exists(f"{S2_CK}/ckpt_latest.pt") and os.path.exists(f"{S2_DRIVE}/ckpt_latest.pt"):
    restore(f"{S2_DRIVE}/ckpt_latest.pt", f"{S2_CK}/ckpt_latest.pt")
if os.path.exists(f"{S2_CK}/ckpt_latest.pt"):
    start_arg = f"--resume {S2_CK}/ckpt_latest.pt"; print("RESUMING the 2B-token fine-tuning run")
else:
    start_arg = "--init-from artifacts/base/base.pt"; print("fresh 2B-token fine-tuning run from the base model")

_stop2, _seen2 = threading.Event(), {}
def _backup2():
    for n in ("ckpt_latest.pt", "ckpt_best.pt", "ckpt_final.pt"):
        f = f"{S2_CK}/{n}"
        if os.path.exists(f):
            key = (os.stat(f).st_mtime, os.stat(f).st_size)
            if _seen2.get(n) != key:
                try: copy_verified(f, S2_DRIVE); _seen2[n] = key
                except Exception as e: print("backup failed (will retry):", e)
threading.Thread(target=lambda: [_backup2() for _ in iter(lambda: _stop2.wait(600), True)], daemon=True).start()
try:
    !tp-train --config configs/run_sft2b.yaml {start_arg}
finally:
    _stop2.set(); _backup2(); print("checkpoints backed up to", S2_DRIVE)

### E4. Compare the base model and the fine-tuned model on the same prompts

Prompts end where a *response* should begin (the end of a docstring, `<answer>`, `<commit_after>`). The fine-tuned model should stop on its own and follow the formats.

In [ ]:
import subprocess, re
LOG = re.compile(r"^\d\d:\d\d:\d\d \| (INFO|WARNING|ERROR|DEBUG)")

def gen(prompt, ckpt, cfg_path="configs/run_sft2b.yaml", max_new_tokens=200, temperature=0.2):
    r = subprocess.run(["tp-generate", "--config", cfg_path, "--checkpoint", ckpt, "--prompt", prompt,
                        "--strategy", "sample", "--temperature", str(temperature), "--top-k", "50",
                        "--max-new-tokens", str(max_new_tokens)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout or r.stderr).splitlines() if not LOG.match(l))

tests = {
 "function": '<file_sep>import re\n\n\ndef extract_emails(text: str) -> list:\n    """Return all email addresses found in text."""',
 "q&a":      '<question>How do I merge two dictionaries in Python?<answer>',
 "edit":     '<commit_before>def mean(xs):\n    return sum(xs) / (len(xs) - 1)\n<commit_message>Fix off-by-one in mean<commit_after>',
 "contest":  '<question>Read two integers a and b from one line and print their sum.<answer>',
}
FT = f"{cfg.train.output_dir}/checkpoints/ckpt_final.pt"
if not os.path.exists(FT): FT = f"{cfg.train.output_dir}/checkpoints/ckpt_best.pt"
for name, p in tests.items():
    print(f"################ {name} ################")
    print("--- BASE ---\n" + gen(p, "artifacts/base/base.pt")[:900])
    print("--- FINE-TUNED (2B tokens) ---\n" + gen(p, FT)[:900] + "\n")

---
### Measure it: pass rates over 30 samples (base vs fine-tuned)

One answer per prompt proves nothing; 30 samples each gives a rough pass rate. Run this after D4 (or E4), which define `gen`, `tests` and the checkpoint paths.
It cuts the **function** answers at the end of the first function for both models (the base model never learned to stop, and stray extra code would fail it unfairly), runs the model-written code (toy prompts only!) and prints two failures per case so you can see *why* they fail. ~15-20 minutes.
A gap of a few out of 30 is noise; look for clear differences (e.g. 4/30 vs 11/30).

In [ ]:
import re, subprocess, sys

FT_CKPT = globals().get("FT", globals().get("SFT_BEST"))      # Part E's checkpoint if you ran E, else Part D's

def after(out, marker):
    i = out.rfind(marker)
    return out[i + len(marker):] if i >= 0 else out

def extract_code(text):
    m = re.search(r"```(?:python)?\n(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def first_function(out):
    lines, seen = [], False
    for ln in out.splitlines():
        if ln.startswith("def "):
            if seen: break
            seen = True
        elif seen and ln.strip() and not ln[0].isspace():
            break
        lines.append(ln)
    return "\n".join(lines) + "\n"

def contest_ok(out):
    code = extract_code(after(out, "print their sum."))
    try:
        r = subprocess.run([sys.executable, "-c", code], input="3 4\n", capture_output=True, text=True, timeout=5)
        return r.stdout.strip() == "7"
    except Exception:
        return False

def edit_ok(out):
    fixed = after(out, "Fix off-by-one in mean")
    return "def mean" in fixed and "len(xs)" in fixed and "len(xs) - 1" not in fixed and "len(xs)-1" not in fixed

def function_ok(out):
    code = after(out, '"""Return all email addresses found in text."""')
    ns = {}
    try:
        exec("import re\n\n\ndef extract_emails(text: str) -> list:\n" + code.rstrip() + "\n", ns)
        return ns["extract_emails"]("write a@b.com or c.d@e.org now") == ["a@b.com", "c.d@e.org"]
    except Exception:
        return False

N = 30
for name, ok, trim in (("function", function_ok, True), ("edit", edit_ok, False), ("contest", contest_ok, False)):
    for label, ckpt in (("BASE", "artifacts/base/base.pt"), ("FINE-TUNED", FT_CKPT)):
        outs = [gen(tests[name], ckpt, temperature=0.5) for _ in range(N)]
        if trim: outs = [first_function(o) for o in outs]
        bad = [o for o in outs if not ok(o)]
        print(f"{name:<9}{label:<11}{N - len(bad)}/{N} pass")
        for o in bad[:2]: print("   example failure:", repr(o[-300:]))

---
# PART F — real benchmarks: HumanEval (164 problems) and MBPP (500 problems)

`scripts/eval_code_benchmarks.py` runs your model on the standard problems and reports **pass@1 with a 95% confidence interval**, then compares two models on the same problems.
Compared with a hand-written test cell it: **(1)** generates inside one process (the model loads once, not once per problem), **(2)** turns the **repetition penalty off** (the old default of 1.1 compounds with every repeated token and wrecks code),
**(3)** runs the generated code in a limited subprocess (time limit, memory limit, destructive calls disabled), and **(4)** tells you whether a difference between two models is real or noise.

Use a disposable runtime: model-written code is executed. Greedy decoding (`--temperature 0`) gives pass@1 directly. **Time (my estimate, not measured): roughly 5-15 minutes per model for HumanEval and 15-40 for MBPP.**
Run it **after SETUP with the latest zip** so the repetition-penalty fix is in place.

In [ ]:
os.makedirs("artifacts/eval", exist_ok=True)
FT_CKPT = globals().get("FT", globals().get("SFT_BEST"))                 # fine-tuned checkpoint from Part E, else Part D
CHECKPOINTS = {"base": "artifacts/base/base.pt", "finetuned": FT_CKPT}
BENCH = ["humaneval", "mbpp"]
CFG = "configs/code_500m_20b.yaml"                                        # same model + tokenizer as every checkpoint here

for name, ckpt in CHECKPOINTS.items():
    for b in BENCH:
        !python scripts/eval_code_benchmarks.py generate --config {CFG} --checkpoint {ckpt} --benchmark {b} \
            --out artifacts/eval/{b}_{name}.jsonl --temperature 0 --max-new-tokens 300

### F2. Score and compare

Each `score` prints pass@1 with its 95% interval and **why the failures failed** (AssertionError = wrong answer, SyntaxError/IndentationError = broken code, NameError = invented names, timeout = too slow).
`compare` then says whether the fine-tuned model is better, worse, or **no clear difference** (the interval includes 0) on the same problems.

In [ ]:
for b in BENCH:
    for name in CHECKPOINTS:
        !python scripts/eval_code_benchmarks.py score --benchmark {b} --completions artifacts/eval/{b}_{name}.jsonl \
            --out artifacts/eval/{b}_{name}.results.jsonl
    print(f"\n===== {b}: base vs fine-tuned =====")
    !python scripts/eval_code_benchmarks.py compare artifacts/eval/{b}_base.results.jsonl artifacts/eval/{b}_finetuned.results.jsonl

import glob
for f in glob.glob("artifacts/eval/*.jsonl"):
    copy_verified(f, f"{DRIVE}/eval")

### F3. (Optional) pass@10 for one model: "best of 10 answers"

Samples 10 answers per problem at a low temperature and reports pass@1/5/10. This is the number that shows what the generate-and-test approach can do for a demo. It takes about 10x longer than greedy.

In [ ]:
!python scripts/eval_code_benchmarks.py generate --config {CFG} --checkpoint {FT_CKPT} --benchmark humaneval \
    --out artifacts/eval/humaneval_finetuned_n10.jsonl --n 10 --temperature 0.4 --max-new-tokens 300
!python scripts/eval_code_benchmarks.py score --benchmark humaneval --completions artifacts/eval/humaneval_finetuned_n10.jsonl \
    --out artifacts/eval/humaneval_finetuned_n10.results.jsonl